In [3]:
import os
import time
import pandas as pd

from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster

In [4]:
TEAM_STATS_DATA_DIR = "data/team_stats"
TEAM_STATS_RAW_DIR = os.path.join(TEAM_STATS_DATA_DIR, "raw")
ROSTER_DATA_DIR = "data/roster"
ROSTER_RAW_DIR = os.path.join(ROSTER_DATA_DIR, "raw")
os.makedirs(TEAM_STATS_RAW_DIR, exist_ok=True)
os.makedirs(ROSTER_RAW_DIR, exist_ok=True)

SEASON_END_YEARS = [2022, 2023, 2024, 2025, 2026]

CRAWL_DELAY = 3  # seconds between actually-fetched (non-cached) nba_api calls

NBA_TEAMS = teams.get_teams()  # 30 dicts: id, full_name, abbreviation, nickname, city, ...

# nba_api's abbreviation 
NBA_API_TO_APP_ABBREV = {"BKN": "BRK", "CHA": "CHO", "PHX": "PHO"}


def app_abbrev(nba_api_abbrev):
    return NBA_API_TO_APP_ABBREV.get(nba_api_abbrev, nba_api_abbrev)


def season_label(end_year):
   
    start = str(end_year - 1)[-2:]
    end = str(end_year)[-2:]
    return f"{start}-{end}"


def nba_api_season_param(end_year):
    # passed into nba_api calls 
    return f"{end_year - 1}-{str(end_year)[-2:]}"


# team_id -> app-canonical abbreviation, built once from NBA_TEAMS
TEAM_ID_TO_ABBREV = {t["id"]: app_abbrev(t["abbreviation"]) for t in NBA_TEAMS}

In [5]:
MEASURE_TYPES = ["Base", "Advanced", "Opponent"]

def fetch_team_stats_json(end_year, measure_type, retries=3, retry_delay=5):
  save_path = os.path.join(TEAM_STATS_RAW_DIR, f"{end_year}_{measure_type}.json")
  if os.path.exists(save_path):
    return save_path
  
  for attempt in range(1, retries +1):
    try:
      resp = leaguedashteamstats.LeagueDashTeamStats(
        season=nba_api_season_param(end_year),
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure_type,
        per_mode_detailed="PerGame" #defaults at season total not per-game

      )
      with open(save_path, "w") as f:
        f.write(resp.get_json())
      return save_path
    except Exception as e:
      print(f"Error fetching {end_year}/{measure_type}: {e} (attempt {attempt}/{retries})")
      if attempt < retries:
        time.sleep(retry_delay * attempt)

  raise RuntimeError(f"Failed to fetch team stats {end_year}/{measure_type} after {retries} attempts")

for end_year in SEASON_END_YEARS:
  for measure_type in MEASURE_TYPES:
    save_path = os.path.join(TEAM_STATS_RAW_DIR, f"{end_year}_{measure_type}.json")
    already_cached = os.path.exists(save_path)

    path = fetch_team_stats_json(end_year, measure_type)
    print(f"{season_label(end_year)} {measure_type}: {path}")

    if not already_cached:
      time.sleep(CRAWL_DELAY)

21-22 Base: data/team_stats/raw/2022_Base.json
21-22 Advanced: data/team_stats/raw/2022_Advanced.json
21-22 Opponent: data/team_stats/raw/2022_Opponent.json
22-23 Base: data/team_stats/raw/2023_Base.json
22-23 Advanced: data/team_stats/raw/2023_Advanced.json
22-23 Opponent: data/team_stats/raw/2023_Opponent.json
23-24 Base: data/team_stats/raw/2024_Base.json
23-24 Advanced: data/team_stats/raw/2024_Advanced.json
23-24 Opponent: data/team_stats/raw/2024_Opponent.json
24-25 Base: data/team_stats/raw/2025_Base.json
24-25 Advanced: data/team_stats/raw/2025_Advanced.json
24-25 Opponent: data/team_stats/raw/2025_Opponent.json
25-26 Base: data/team_stats/raw/2026_Base.json
25-26 Advanced: data/team_stats/raw/2026_Advanced.json
25-26 Opponent: data/team_stats/raw/2026_Opponent.json
